# 06 - EDA: VinDr-CXR (subconjunto Tuberculosis)

El dataset completo de VinDr-CXR (PhysioNet, credenciales requeridas) pesa **141.2 GB comprimido / 191.8 GB
descomprimido** en DICOM. No fue posible descargarlo completo — solo se descargaron los archivos CSV de
anotaciones, sin las imágenes.

Antes de descargar nada, se analizaron `image_labels_train.csv` (45,000 filas = 15,000 imágenes x 3
radiólogos) e `image_labels_test.csv` (3,000 filas = 3,000 imágenes, ya consenso de 5 radiólogos) para
decidir qué vale la pena descargar. Resultado: **12,652 Normal / 717 Pneumonia / 646 Tuberculosis**
(verificado contra los conteos normal/anormal publicados oficialmente por PhysioNet). Como VinDr-CXR no
tiene casos de COVID-19 y ya había margen de sobra en Normal/Pneumonia desde otras fuentes, se decidió
descargar **solo las 646 imágenes de Tuberculosis** (482 train + 164 test), una por una vía `wget` (las
imágenes se sirven individualmente en PhysioNet, no en un solo archivo empaquetado).

Este notebook reconstruye esa selección de forma reproducible (no depende del script puntual que generó
la lista de descarga) y hace el mismo análisis de calidad/hash/demografía que las demás fuentes DICOM
(mismo patrón que RSNA).

In [ ]:
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

sys.path.insert(0, ".")
import eda_utils as eu

tqdm.pandas()

DATASET_NAME = "VinDr_CXR"
DATASET_DIR = Path("../Datasets/pyshionet")
TRAIN_DIR = DATASET_DIR / "images/train"
TEST_DIR = DATASET_DIR / "images/test"
LABELS_TRAIN_CSV = DATASET_DIR / "anotations/image_labels_train.csv"
LABELS_TEST_CSV = DATASET_DIR / "anotations/image_labels_test.csv"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

OUTPUT_DIR = Path("../resultados/eda") / DATASET_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HASH_DIR = Path("../Datasets/_eda_hashes")
HASH_DIR.mkdir(parents=True, exist_ok=True)

## 1. Reconstrucción de la selección (voto mayoritario / consenso)

`image_labels_train.csv` trae 3 filas por imagen (una por radiólogo) — para una etiqueta por imagen se
necesita voto mayoritario (>=2 de 3 marcan Tuberculosis). `image_labels_test.csv` ya viene una fila por
imagen (consenso de 5 radiólogos), se usa directo.

In [ ]:
labels_train = pd.read_csv(LABELS_TRAIN_CSV)
labels_test = pd.read_csv(LABELS_TEST_CSV)
print(f"Filas train (3 x imagen): {len(labels_train)} -> {labels_train['image_id'].nunique()} imágenes únicas")
print(f"Filas test (ya consenso, 1 x imagen): {len(labels_test)}")

# Voto mayoritario en train: >=2 de 3 radiólogos marcan Tuberculosis=1
tb_votes_train = labels_train.groupby("image_id")["Tuberculosis"].sum()
tb_train_ids = tb_votes_train[tb_votes_train >= 2].index
print(f"Tuberculosis por voto mayoritario en train: {len(tb_train_ids)}")

# Test: ya es consenso, se usa directo
tb_test_ids = labels_test.loc[labels_test["Tuberculosis"] == 1, "image_id"]
print(f"Tuberculosis por consenso en test: {len(tb_test_ids)}")

print(f"Total Tuberculosis seleccionado: {len(tb_train_ids) + len(tb_test_ids)}")

# Conteos informativos de las otras categorías relevantes del proyecto (NO se descargaron, solo para
# documentar la decisión de no bajarlas)
normal_train = (labels_train.groupby("image_id")["No finding"].sum() >= 2).sum()
normal_test = (labels_test["No finding"] == 1).sum()
pne_train = (labels_train.groupby("image_id")["Pneumonia"].sum() >= 2).sum()
pne_test = (labels_test["Pneumonia"] == 1).sum()
print(f"(informativo, no descargado) Normal: {normal_train + normal_test} | Pneumonia: {pne_train + pne_test}")

In [ ]:
df_train = pd.DataFrame({"image_id": tb_train_ids, "split": "train"})
df_test = pd.DataFrame({"image_id": tb_test_ids, "split": "test"})
df = pd.concat([df_train, df_test], ignore_index=True)
df["target_class"] = "Tuberculosis"
df["path"] = df.apply(
    lambda r: str((TRAIN_DIR if r["split"] == "train" else TEST_DIR) / f"{r['image_id']}.dicom"), axis=1
)
print(f"Total reconstruido: {len(df)}")
df["split"].value_counts()

### Verificación contra los archivos descargados

La lista `tuberculosis_urls.txt` que se usó para el `wget` selectivo se generó con esta misma lógica
(script puntual, no notebook) — aquí se verifica que lo descargado coincide 100% con lo reconstruido.

In [ ]:
downloaded = {p.stem: p for p in list(TRAIN_DIR.glob("*.dicom")) + list(TEST_DIR.glob("*.dicom"))}
print(f"Archivos físicos descargados: {len(downloaded)}")

df["existe"] = df["image_id"].isin(downloaded)
faltantes = df[~df["existe"]]
sobrantes = set(downloaded) - set(df["image_id"])
print(f"Filas reconstruidas sin archivo local: {len(faltantes)}")
print(f"Archivos locales que no están en la reconstrucción: {len(sobrantes)}")
assert len(faltantes) == 0 and len(sobrantes) == 0, "Descarga incompleta o inconsistente con la selección"
print("Coincidencia exacta: 646/646 (482 train + 164 test).")

## 2. Conteo por clase

Solo hay una clase en este subconjunto (por diseño: es lo único que se descargó). El total real se
suma al resto de fuentes de Tuberculosis del proyecto (Chest_x_ray, TBX11K, Shenzhen/Montgomery,
SilicoData) recién en la consolidación, después del cruce de hashes.

In [ ]:
totales = df["target_class"].value_counts()
print(totales)
totales.to_frame("n").to_csv(OUTPUT_DIR / "conteo_por_clase.csv")

ax = df.groupby(["split"]).size().plot(kind="bar", figsize=(5, 4), color="#C44E52")
ax.set_title(f"{DATASET_NAME}: Tuberculosis por split de origen")
ax.set_ylabel("Número de imágenes")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "conteo_por_clase.png", dpi=120)
plt.show()

## 3. Calidad e integridad

Mismo patrón que RSNA: se decodifica cada DICOM una sola vez y se reutiliza el array para dimensiones,
heurísticas de calidad y hash perceptual. Aquí son solo 646 archivos, tarda segundos.

**Nota importante sobre bit depth:** a diferencia de RSNA (DICOM de 8 bits, ya en rango 0-255), VinDr-CXR
trae DICOM de alto rango dinámico real (`BitsStored=14`, valores 0-16383). Convertir ese array directo a
escala de grises de 8 bits (`Image.convert("L")`) **trunca en vez de reescalar** — casi todo píxel queda
saturado a blanco, lo que rompe silenciosamente las heurísticas de calidad y el hash perceptual (se
detectó al ejecutar esta sección la primera vez: marcaba 100% de las imágenes por "brillo" y el phash
salía degenerado con ~38% de duplicados falsos dentro del mismo subconjunto). Por eso aquí se usa
`eu.probe_dicom_normalized` en vez de `eu.probe_dicom`: aplica VOI LUT (windowing con
`WindowCenter`/`WindowWidth` de la cabecera — lo que vería un radiólogo) antes de reescalar a uint8.

In [ ]:
widths, heights, errors = [], [], []
content_ratios, glare_fractions, phashes = [], [], []

for p in tqdm(df["path"], desc="Decodificando DICOM"):
    arr, err = eu.probe_dicom_normalized(p)
    if err is not None:
        widths.append(None); heights.append(None); errors.append(err)
        content_ratios.append(None); glare_fractions.append(None); phashes.append(None)
        continue
    h, w = arr.shape[:2]
    widths.append(w); heights.append(h); errors.append(None)
    thumb = eu.array_to_thumbnail_gray(arr)
    content_ratios.append(eu.content_ratio_from_array(thumb))
    glare_fractions.append(eu.glare_fraction_from_array(thumb))
    phashes.append(eu.phash_from_array(thumb))

df["width"] = widths
df["height"] = heights
df["error"] = errors
df["content_ratio"] = content_ratios
df["glare_fraction"] = glare_fractions
df["phash"] = phashes

corrupt = df[df["error"].notna()]
print(f"Imágenes corruptas / no decodificables: {len(corrupt)}")
corrupt[["path", "error"]]

In [ ]:
print("Dimensiones (ancho x alto):")
df[["width", "height"]].describe()

### 3.1 Candidatas a excluir por calidad de imagen (mismo criterio que las fuentes anteriores)

In [ ]:
CONTENT_RATIO_THRESHOLD = 0.55
GLARE_THRESHOLD = 0.15

calidad = df[df["error"].isna()].copy()
calidad["flag_bordes"] = calidad["content_ratio"] < CONTENT_RATIO_THRESHOLD
calidad["flag_brillo"] = calidad["glare_fraction"] > GLARE_THRESHOLD
calidad["motivo"] = calidad.apply(
    lambda r: ", ".join(
        m for m, f in [("bordes/recorte incompleto", r["flag_bordes"]), ("posible brillo/reflejo", r["flag_brillo"])] if f
    ),
    axis=1,
)
sospechosas = calidad[calidad["flag_bordes"] | calidad["flag_brillo"]].copy()
print(f"Candidatas marcadas para revisión manual: {len(sospechosas)} de {len(calidad)} "
      f"({100 * len(sospechosas) / len(calidad):.1f}%)")
sospechosas["target_class"].value_counts()

In [ ]:
sospechosas[["path", "target_class", "content_ratio", "glare_fraction", "motivo"]].to_csv(
    OUTPUT_DIR / "candidatas_baja_calidad.csv", index=False
)

def _dicom_loader(path):
    arr, _ = eu.probe_dicom_normalized(path)
    return arr

if len(sospechosas):
    eu.show_flagged(sospechosas, save_path=OUTPUT_DIR / "candidatas_baja_calidad_muestra.png", loader=_dicom_loader)
else:
    print("Ninguna candidata para mostrar.")

## 4. Muestra visual

In [ ]:
eu.show_samples(df[df["error"].isna()], save_path=OUTPUT_DIR / "muestra_por_clase.png", loader=_dicom_loader)

## 5. Riesgo de solapamiento / duplicados

Igual que en RSNA, el `md5` solo detecta copias byte-idénticas del propio `.dicom`. El `phash` es lo
que realmente importa aquí: es la pieza que se necesita para el cruce contra las otras 4 fuentes de
Tuberculosis (Chest_x_ray, TBX11K, Shenzhen/Montgomery, SilicoData) en el notebook de cruce de hashes.

In [ ]:
df["md5"] = df["path"].progress_apply(eu.md5_file)
n_dupes_md5 = df["md5"].duplicated().sum()
n_dupes_phash = df["phash"].duplicated().sum()
print(f"Duplicados exactos (MD5) dentro de este mismo subconjunto: {n_dupes_md5}")
print(f"Duplicados por phash dentro de este mismo subconjunto: {n_dupes_phash}")

In [ ]:
df[["path", "md5", "phash", "target_class", "split"]].assign(source_dataset=DATASET_NAME).to_csv(
    HASH_DIR / f"{DATASET_NAME}_md5.csv", index=False
)

## 6. Metadata demográfica

La cabecera DICOM de VinDr-CXR solo trae `PatientAge` y `PatientSex` (no `ViewPosition`, `Modality` ni
`BodyPartExamined` como en RSNA — VinDr no las incluye, se confirmó revisando la cabecera completa de un
archivo de muestra antes de escribir esta celda).

In [ ]:
meta_records = [eu.dicom_metadata(p, tags=("PatientAge", "PatientSex")) for p in tqdm(df["path"], desc="Leyendo metadata DICOM")]
meta_df = pd.DataFrame(meta_records)
df = pd.concat([df.reset_index(drop=True), meta_df.reset_index(drop=True)], axis=1)

df["PatientAge"] = df["PatientAge"].str.extract(r"(\d+)").astype(float)
print("Edad del paciente:")
display(df["PatientAge"].describe())
print()
print("Sexo del paciente:")
display(df["PatientSex"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
df["PatientAge"].plot(kind="hist", bins=20, ax=ax, color="#C44E52")
ax.set_title(f"{DATASET_NAME}: distribución de edad (Tuberculosis)")
ax.set_xlabel("Edad")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "demografia_por_clase.png", dpi=120)
plt.show()

## 7. Veredicto

- Solo se descargó el subconjunto de **Tuberculosis** (646 imágenes: 482 train + 164 test), reconstruido
  aquí con la misma lógica de voto mayoritario (train) / consenso (test) que se usó para generar la lista
  de descarga — coincide exactamente con lo descargado (646/646, verificado en la Sección 1).
- No se descargó Normal (12,652) ni Pneumonia (717) de esta fuente: no aportaban nada nuevo dado el
  techo de 6,000 ya cubierto por otras fuentes, y VinDr-CXR no tiene casos de COVID-19.
- Cero imágenes corruptas / no decodificables.
- Primera fuente DICOM de alto rango dinámico real (14 bits) del proyecto — requirió normalización con
  VOI LUT (`eu.probe_dicom_normalized`) antes de calcular calidad/hash; usar la ruta genérica de 8 bits
  (`eu.probe_dicom`, la que sirve para RSNA) rompía silenciosamente las heurísticas (ver Sección 3).
- Metadata demográfica real disponible (edad, sexo) directo de la cabecera DICOM — sin `ViewPosition`
  ni `PatientID` (VinDr no los incluye en estos archivos).
- `phash` calculado y exportado a `_eda_hashes/VinDr_CXR_md5.csv`, listo para el cruce contra las otras
  4 fuentes de Tuberculosis en el notebook de cruce de hashes.
- Aporta como techo real: hasta 646 imágenes nuevas de Tuberculosis, pendiente de saber cuántas son
  realmente nuevas tras el cruce de hashes (mismos pacientes/estudios podrían, en teoría, superponerse
  con TBX11K, que también usa una submuestra de un dataset vietnamita público de tórax).